# SuperFRAME: 12-Hour High-Throughput Kaggle Training (Dual Tesla T4)
### Towards Beating FUSED on Forensic Inpainting Detection & Localization

**Notebook Requirements:**
- **Accelerator:** GPU T4 x2 *(Kaggle sidebar: Settings → Accelerator → GPU T4 x2)*
- **Internet:** ON *(Settings → Internet → On)*
- **Runtime:** Up to 12 Hours (runs in background with 'Save & Run All')

In [ ]:
# Step 1: Check GPUs and Clone GitHub Repository
!nvidia-smi

!rm -rf ffdm
!git clone https://github.com/Dakshankarthic/ffdm.git
%cd ffdm

# Install dependencies
!pip install -q timm pyarrow scikit-learn huggingface_hub scipy

In [ ]:
# Step 2: Verify PyTorch Multi-GPU Setup
import torch

print(f"PyTorch Version: {torch.__version__}")
num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB)")

assert num_gpus >= 1, "GPU accelerator not detected! Enable GPU T4 x2 in notebook settings."

In [ ]:
# Step 3: Run High-Throughput Joint Dual-View Training
# - 50% Genuine, 50% Edited Balanced Batches
# - Resized Whole-Image (384x384) View + Native-Resolution (384x384) Crops
# - DataParallel across both Tesla T4 GPUs (Batch Size 16 per GPU = 32 Total)
# - Saves best checkpoint based on validation Balanced Accuracy

import sys
sys.path.append("controlled_localization/src")

!python controlled_localization/src/controlled_localization/train_kaggle_dual_t4.py \
    --steps 15000 \
    --batch_size 16 \
    --val_interval 500

In [ ]:
# Step 4: Evaluate Checkpoint with Separated Thresholds on the 100-Image Benchmark
!python controlled_localization/src/controlled_localization/eval_score_distributions.py

In [ ]:
# Step 5: Display Multi-Paper Benchmark Comparison Table
import json
from pathlib import Path

audit_p = Path("controlled_localization/reports/experiment1_score_calibration_audit.json")
if audit_p.exists():
    with open(audit_p, "r") as f:
        data = json.load(f)
    print("=" * 80)
    print("SUPERFRAME BENCHMARK AUDIT (CALIBRATED THRESHOLDS):")
    print("=" * 80)
    for m_name, res in data.items():
        print(f"\nModel: {m_name}")
        for stype in ["whole", "pool", "hybrid"]:
            if stype in res.get("frozen_test_results", {}):
                t = res["frozen_test_results"][stype]
                print(f"  - {stype:<7}: Acc = {t['accuracy']:.2f}% | Balanced Acc = {t['balanced_accuracy']:.2f}% | AUROC = {t['auroc']:.2f}% | F1 = {t['f1']:.2f}%")
                print(f"    Confusion (TP/TN/FP/FN): {t['confusion']['tp']}/{t['confusion']['tn']}/{t['confusion']['fp']}/{t['confusion']['fn']}")
else:
    print("Evaluation audit report not found.")

In [ ]:
# Step 6: Package Trained Checkpoints and Reports for Download
!mkdir -p /kaggle/working/superframe_artifacts
!cp -r controlled_localization/reports /kaggle/working/superframe_artifacts/
!cp -r /kaggle/working/output/* /kaggle/working/superframe_artifacts/ 2>/dev/null || true
!zip -r /kaggle/working/superframe_kaggle_artifacts.zip /kaggle/working/superframe_artifacts
print("Artifacts successfully packaged to: /kaggle/working/superframe_kaggle_artifacts.zip")